# Asistente de descripciones comerciales
Completa la ficha para generar una propuesta. Las alertas son orientativas: revisa los datos antes de usarla. Ejecuta las secciones en orden con GPU T4.


## 1. Instalar dependencias

Se usan las versiones registradas durante la evaluación del adaptador. Si Colab solicita reiniciar el entorno, reinícialo y vuelve a ejecutar desde esta celda.

In [ ]:
%pip install -q "transformers==5.22.1" "peft==0.21.2" accelerate safetensors requests ipywidgets

## 2. Descargar y verificar los adaptadores

Se descargan las tres partes publicadas en GitHub, fijadas a una versión del repositorio, y se verifica la huella del paquete antes de cargar sus archivos. Si ya tienes el ZIP completo en Colab, indica su ruta en `ZIP_LOCAL`.

In [ ]:
from pathlib import Path
import hashlib
import json
import zipfile
import requests

ZIP_LOCAL = None
CARPETA = Path('/content/pipeline_producto')
CARPETA.mkdir(parents=True, exist_ok=True)
paquete = Path(ZIP_LOCAL) if ZIP_LOCAL else CARPETA / 'docente_desarrollado.zip'
huella = 'bf022a8cfa015a2745b6945def269c386b4adb5f6985c66a4e1a07d48c72b85e'
revision_repo = 'bd59bb08c04247d75e955f6516a83a3c29bf5501'

if not paquete.exists():
    url_descarga = f'https://raw.githubusercontent.com/isaacHeYe/PLN_final_products_destilation/{revision_repo}'
    with paquete.open('wb') as destino:
        for parte in range(1, 4):
            url = f'{url_descarga}/docente_desarrollado.zip.parte{parte:02d}'
            with requests.get(url, stream=True, timeout=120) as respuesta:
                respuesta.raise_for_status()
                for bloque in respuesta.iter_content(1024 * 1024):
                    destino.write(bloque)
            print(f'Parte {parte}/3 descargada')

if hashlib.sha256(paquete.read_bytes()).hexdigest() != huella:
    raise ValueError('La huella del paquete no coincide. Elimina el archivo y vuelve a descargarlo.')

archivos = ['resultados/configuracion.json', 'resultados/soft_lora/mejor/soft.pt', 'resultados/soft_lora/mejor/lora/adapter_config.json', 'resultados/soft_lora/mejor/lora/adapter_model.safetensors']
with zipfile.ZipFile(paquete) as archivo:
    for nombre in archivos:
        destino = CARPETA / nombre
        destino.parent.mkdir(parents=True, exist_ok=True)
        destino.write_bytes(archivo.read(nombre))

config = json.loads((CARPETA / 'resultados/configuracion.json').read_text())
print('Adaptadores verificados y preparados.')

## 3. Cargar el modelo

Se fija la revisión del modelo base usada en el experimento. La carga requiere GPU con suficiente memoria; el modelo base se descarga desde Hugging Face.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import sys
import peft.import_utils

original_torchao = peft.import_utils.is_torchao_available
for nombre, modulo in list(sys.modules.items()):
    if nombre.startswith('peft.') and modulo is not None and vars(modulo).get('is_torchao_available') is original_torchao:
        vars(modulo)['is_torchao_available'] = lambda: False

if not torch.cuda.is_available():
    raise RuntimeError('Activa una GPU en el entorno de ejecución de Colab.')

modelo_id = config['modelo']
revision = config['revision']
tok = AutoTokenizer.from_pretrained(modelo_id, revision=revision, trust_remote_code=False)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(modelo_id, revision=revision, dtype=torch.float16, device_map={'': 0}, trust_remote_code=False)
carpeta_mejor = CARPETA / 'resultados/soft_lora/mejor'
modelo = PeftModel.from_pretrained(base, carpeta_mejor / 'lora', is_trainable=False)
modelo.eval()
soft = torch.load(carpeta_mejor / 'soft.pt', map_location='cuda', weights_only=True)
if soft.ndim != 2 or soft.shape[0] != config['tokens_suaves'] or soft.shape[1] != modelo.get_input_embeddings().weight.shape[1]:
    raise ValueError('Los vectores suaves no coinciden con el modelo y su configuración.')
print(f'Modelo cargado en {torch.cuda.get_device_name(0)}; vectores suaves: {soft.shape[0]}.')

## 4. Preparar y validar la ficha

Se eliminan etiquetas HTML y espacios sobrantes. Para generar se requiere un título y una característica diferente del título. El lenguaje potencialmente ofensivo se señala para que el usuario lo revise: la lista es limitada, puede dar falsas alarmas y no sustituye una revisión contextual.

El filtro usa la lista española [LDNOOBW de Shutterstock y colaboradores](https://github.com/LDNOOBW/List-of-Dirty-Naughty-Obscene-and-Otherwise-Bad-Words), versión `5faf2ba42d7b1c0977169ec3611df25a3c08eb13`, licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), junto con términos adicionales del proyecto. Incluye términos que pueden ser legítimos, como nombres de herramientas o identidades; son alertas, no juicios automáticos.


In [ ]:
import html
import re
import unicodedata
from html.parser import HTMLParser

class TextoHTML(HTMLParser):
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.partes = []

    def handle_data(self, dato):
        self.partes.append(dato)

    def handle_starttag(self, tag, attrs):
        if tag in {'br', 'p', 'li', 'div'}:
            self.partes.append(' ')

    def handle_endtag(self, tag):
        if tag in {'p', 'li', 'div'}:
            self.partes.append(' ')

def limpiar_texto(valor):
    texto = '' if valor is None else str(valor)
    parser = TextoHTML()
    parser.feed(texto)
    texto = html.unescape(''.join(parser.partes))
    texto = unicodedata.normalize('NFC', texto)
    return re.sub(r'\s+', ' ', texto).strip()

def normalizar(texto):
    texto = unicodedata.normalize('NFKD', texto.casefold())
    return ''.join(c for c in texto if not unicodedata.combining(c))

LEXICO_LDNOOBW = ('Asesinato', 'asno', 'bastardo', 'Bollera', 'Cabrón', 'Caca', 'Chupada', 'Chupapollas', 'Chupetón', 'concha', 'Concha de tu madre', 'Coño', 'Coprofagía', 'Culo', 'Drogas', 'Esperma', 'Fiesta de salchichas', 'Follador', 'Follar', 'Gilipichis', 'Gilipollas', 'Hacer una paja', 'Haciendo el amor', 'Heroína', 'Hija de puta', 'Hijaputa', 'Hijo de puta', 'Hijoputa', 'Idiota', 'Imbécil', 'infierno', 'Jilipollas', 'Kapullo', 'Lameculos', 'Maciza', 'Macizorra', 'maldito', 'Mamada', 'Marica', 'Maricón', 'Mariconazo', 'martillo', 'Mierda', 'Nazi', 'Orina', 'Pedo', 'Pendejo', 'Pervertido', 'Pezón', 'Pinche', 'Pis', 'Prostituta', 'Puta', 'Racista', 'Ramera', 'Sádico', 'Semen', 'Sexo', 'Sexo oral', 'Soplagaitas', 'Soplapollas', 'Tetas grandes', 'Tía buena', 'Travesti', 'Trio', 'Verga', 'vete a la mierda', 'Vulva')
TERMINOS_ADICIONALES = ('mierda', 'puta', 'puto', 'joder', 'imbecil', 'estupido', 'estupida', 'cabron', 'cabrona')

def lenguaje_sensible(texto):
    texto = normalizar(texto)
    terminos = sorted({normalizar(x.strip()) for x in (*LEXICO_LDNOOBW, *TERMINOS_ADICIONALES) if x.strip()}, key=lambda x: (-len(x), x))
    return [palabra for palabra in terminos if re.search(r'(?<!\w)' + re.escape(palabra).replace(r'\ ', r'\s+') + r'(?!\w)', texto)]

def limpiar_ficha(ficha):
    return {campo: limpiar_texto(ficha.get(campo, '')) for campo in ('titulo', 'marca', 'color', 'bullets')}

def validar_ficha(ficha):
    errores = []
    alertas = []
    if not ficha['titulo']:
        errores.append('Falta el título del producto.')
    if not ficha['bullets'] or normalizar(ficha['bullets']) == normalizar(ficha['titulo']):
        errores.append('Añade al menos una característica adicional al título.')
    sensibles = lenguaje_sensible(' '.join(ficha.values()))
    if sensibles:
        alertas.append('Revisa el posible lenguaje inapropiado en la ficha: ' + ', '.join(sensibles) + '. Puede haber falsos positivos.')
    capacidades = set(re.findall(r'(?<!\w)\d+(?:[.,]\d+)?\s*(?:gb|tb)(?!\w)', normalizar(ficha['titulo'] + ' ' + ficha['bullets'])))
    capacidades = {re.sub(r'\s+', '', valor) for valor in capacidades}
    if len(capacidades) > 1:
        alertas.append('Se encontraron varias capacidades en GB o TB; verifica si son variantes o datos contradictorios: ' + ', '.join(sorted(capacidades)) + '.')
    return errores, alertas

def formatear_ficha(ficha):
    return '\n'.join(f'{campo}: {ficha[campo]}' for campo in ('titulo', 'marca', 'color', 'bullets'))

## 5. Generación y controles

La generación usa la instrucción y la configuración del experimento. Los controles comparan números, algunas características y palabras sensibles; también señalan posibles garantías o promesas. Son heurísticas limitadas y pueden dar falsas alarmas o pasar por alto errores. El texto generado se conserva para compararlo con la versión editada.

In [ ]:
def generar(ficha):
    mensajes = [{'role': 'system', 'content': config['instruccion']}, {'role': 'user', 'content': formatear_ficha(ficha)}]
    ids = tok.apply_chat_template(mensajes, tokenize=True, add_generation_prompt=True, enable_thinking=False)
    ids = ids['input_ids'] if hasattr(ids, 'keys') else ids
    if len(ids) + soft.shape[0] + config['max_nuevos'] > config['max_tokens_total']:
        raise ValueError('La ficha supera el límite de contexto; acórtala manualmente sin quitar datos importantes.')
    ids = torch.tensor([ids], device='cuda')
    emb = modelo.get_input_embeddings()(ids)
    emb = torch.cat((soft.to(emb.dtype).unsqueeze(0), emb), dim=1)
    mascara = torch.ones(emb.shape[:2], dtype=torch.long, device='cuda')
    resultado = []
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        salida = modelo(inputs_embeds=emb, attention_mask=mascara, use_cache=True)
        cache = salida.past_key_values
        siguiente = salida.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        for _ in range(config['max_nuevos']):
            valor = int(siguiente.item())
            resultado.append(valor)
            if valor == tok.eos_token_id:
                break
            mascara = torch.cat((mascara, torch.ones((1, 1), dtype=torch.long, device='cuda')), dim=1)
            salida = modelo(input_ids=siguiente, attention_mask=mascara, past_key_values=cache, use_cache=True)
            cache = salida.past_key_values
            siguiente = salida.logits[:, -1, :].argmax(dim=-1, keepdim=True)
    return tok.decode(resultado, skip_special_tokens=True).strip(), bool(resultado and resultado[-1] == tok.eos_token_id)

def numeros(texto):
    return set(re.findall(r'(?<!\w)\d+(?:[.,]\d+)?(?!\w)', normalizar(texto)))

def controles(ficha, texto, finalizada):
    alertas = []
    fuente = formatear_ficha(ficha)
    if not texto:
        alertas.append('La descripción está vacía.')
    if not finalizada:
        alertas.append('La descripción puede estar incompleta: se alcanzó el límite de generación.')
    ajenos = numeros(texto) - numeros(fuente)
    if ajenos:
        alertas.append('Revisar números que no aparecen literalmente en la ficha: ' + ', '.join(sorted(ajenos)) + '.')
    faltantes = numeros(fuente) - numeros(texto)
    if faltantes:
        alertas.append('Revisar si se omitieron datos numéricos de la ficha: ' + ', '.join(sorted(faltantes)) + '.')
    sensibles = lenguaje_sensible(texto)
    if sensibles:
        alertas.append('Revisar posible lenguaje inapropiado en la descripción: ' + ', '.join(sensibles) + '.')
    if len(texto.split()) < 15:
        alertas.append('Descripción corta: revisa si conserva las características importantes.')
    texto_n = normalizar(texto)
    fuente_n = normalizar(fuente)
    for expresion in ('garantia', 'garantizado', 'garantizada', 'certificado', 'certificada'):
        if re.search(r'(?<!\w)' + expresion + r'(?!\w)', texto_n) and not re.search(r'(?<!\w)' + expresion + r'(?!\w)', fuente_n):
            alertas.append('Revisar posible afirmación no respaldada: ' + expresion + '.')
    return alertas

def procesar(ficha, aceptar_alertas_entrada=False):
    limpia = limpiar_ficha(ficha)
    errores, alertas_entrada = validar_ficha(limpia)
    if errores:
        return {'estado': 'ficha_insuficiente', 'ficha_limpia': limpia, 'descripcion': None, 'finalizada': None, 'alertas_entrada': alertas_entrada, 'alertas': errores}
    if alertas_entrada and not aceptar_alertas_entrada:
        return {'estado': 'revisar_entrada', 'ficha_limpia': limpia, 'descripcion': None, 'finalizada': None, 'alertas_entrada': alertas_entrada, 'alertas': []}
    try:
        descripcion, finalizada = generar(limpia)
    except ValueError as error:
        return {'estado': 'ficha_no_procesada', 'ficha_limpia': limpia, 'descripcion': None, 'finalizada': None, 'alertas_entrada': alertas_entrada, 'alertas': [str(error)]}
    alertas = controles(limpia, descripcion, finalizada)
    return {'estado': 'revision_humana', 'ficha_limpia': limpia, 'descripcion': descripcion, 'finalizada': finalizada, 'alertas_entrada': alertas_entrada, 'alertas': alertas}

## 6. Crear una propuesta
Completa la ficha, genera la descripción y revisa las alertas antes de usarla.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

estilos = widgets.HTML(value='''<style>
.asistente-productos {max-width: 850px; padding: 26px; border: 1px solid #e4e9f2; border-radius: 18px; background: #f8faff; font-family: system-ui, sans-serif;}
.asistente-productos .widget-label {font-weight: 600; color: #233454;}
.asistente-productos input, .asistente-productos textarea {border: 1px solid #d5deeb !important; border-radius: 9px !important; padding: 10px !important; background: white !important;}
.asistente-productos .alerta-descripcion textarea {color: #b00020 !important; font-weight: 600; background: #fff3f3 !important; border-color: #eaa5aa !important;}
.asistente-productos button {border-radius: 9px !important; font-weight: 600 !important;}
</style>''')
encabezado = widgets.HTML(value='<h2 style="margin:0;color:#172b4d">✨ Crea una descripción comercial</h2><p style="color:#5b6980;margin:6px 0 18px">Cuéntanos sobre tu producto y revisa la propuesta antes de publicarla.</p>')
estilo = {'description_width': 'initial'}
ancho = widgets.Layout(width='100%')
titulo = widgets.Text(description='Producto *', placeholder='Ej. Auriculares inalámbricos', layout=ancho, style=estilo)
marca = widgets.Text(description='Marca', placeholder='Opcional', layout=ancho, style=estilo)
color = widgets.Text(description='Color', placeholder='Opcional', layout=ancho, style=estilo)
caracteristicas = widgets.Textarea(description='Características *', placeholder='Añade materiales, medidas, funciones o compatibilidad.', layout=widgets.Layout(width='100%', height='110px'), style=estilo)
boton_generar = widgets.Button(description='Generar propuesta', button_style='primary', icon='magic', layout=widgets.Layout(width='210px', height='42px'))
estado = widgets.HTML()
propuesta = widgets.Textarea(description='Descripción', placeholder='Tu propuesta aparecerá aquí.', layout=widgets.Layout(width='100%', height='200px'), style=estilo)
alertas_salida = widgets.HTML()
nota = widgets.HTML(value='<p style="color:#65748a;font-size:13px">Revisa siempre los datos y las afirmaciones antes de usar la descripción.</p>')

def al_generar(boton):
    boton.disabled = True
    propuesta.value = ''
    propuesta.remove_class('alerta-descripcion')
    alertas_salida.value = ''
    estado.value = '<span style="color:#4169a1">Preparando tu propuesta…</span>'
    ficha = {'titulo': titulo.value, 'marca': marca.value, 'color': color.value, 'bullets': caracteristicas.value}
    try:
        resultado = procesar(ficha)
        mensajes = list(dict.fromkeys(resultado['alertas_entrada'] + resultado['alertas']))
        if resultado['descripcion'] is None:
            propuesta.add_class('alerta-descripcion')
            propuesta.value = '\n'.join(mensajes) or 'No se pudo generar la propuesta. Revisa la ficha.'
            estado.value = '<span style="color:#b00020">Revisa la información para continuar.</span>'
        else:
            propuesta.value = resultado['descripcion']
            estado.value = '<span style="color:#18744c">Propuesta lista para revisar.</span>'
            if mensajes:
                alertas_salida.value = '<div style="background:#fff7e2;border-radius:9px;padding:12px;color:#765315"><strong>Antes de usarla, revisa:</strong><br>' + '<br>'.join('• ' + html.escape(m) for m in mensajes) + '</div>'
    except Exception as error:
        propuesta.add_class('alerta-descripcion')
        propuesta.value = 'No se pudo generar la propuesta: ' + str(error)
        estado.value = '<span style="color:#b00020">Ocurrió un error.</span>'
    finally:
        boton.disabled = False

boton_generar.on_click(al_generar)
interfaz = widgets.VBox([estilos, encabezado, titulo, marca, color, caracteristicas, boton_generar, estado, propuesta, alertas_salida, nota])
interfaz.add_class('asistente-productos')
display(interfaz)


## 7. Usar la propuesta
Copia o edita la descripción y revisa sus datos antes de publicarla.
